# 第3章 機械学習 ― サンプルコード
『AI実装検定 A級・S級 完全版』の第3章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、pandas 3.0.5、scikit-learn 1.9.1

In [ ]:
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 pandas==3.0.5 scikit-learn==1.9.1
import sys; print(sys.version)


## リスト3.1　train_test_split による分割

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.arange(20).reshape(10, 2)
y = np.array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(Xtr.shape, Xte.shape)
print(ytr.sum(), yte.sum())          # 学習側・テスト側の「1」の数
a, _, _, _ = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(np.array_equal(a, Xtr))        # 同じ random_state なら同じ分割

**本書に載っている出力**
```
(7, 2) (3, 2)
2 1
True
```

## リスト3.2　StandardScaler と MinMaxScaler

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

Xtr = np.array([[1.0], [2.0], [3.0]])
Xte = np.array([[5.0]])

ss = StandardScaler().fit(Xtr)            # 平均・標準偏差は学習データだけから
print(ss.mean_, ss.scale_.round(4))
print(ss.transform(Xte).round(4))         # テストにも「学習データの」値を使う

mm = MinMaxScaler().fit(Xtr)
print(mm.transform(Xte))                  # 範囲外は 1 を超える

**本書に載っている出力**
```
[2.] [0.8165]
[[3.6742]]
[[2.]]
```

## リスト3.3　データ漏洩：テストを混ぜて fit すると

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

Xtr = np.array([[1.0], [2.0], [3.0]])
Xte = np.array([[5.0]])
good = StandardScaler().fit(Xtr).transform(Xte)
bad = StandardScaler().fit(np.vstack([Xtr, Xte])).transform(Xte)   # テストを混ぜて fit
print(good.round(4), bad.round(4))

**本書に載っている出力**
```
[[3.6742]] [[1.5213]]
```

## リスト3.4　エンコーディングと欠損値補完

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
import numpy as np

colors = pd.DataFrame({"color": ["red", "blue", "red", "green"]})
oh = OneHotEncoder(sparse_output=False).fit(colors)
print(oh.categories_[0])
print(oh.transform(colors))
print(OrdinalEncoder().fit_transform(colors).ravel())

X = np.array([[1.0], [np.nan], [3.0]])
print(SimpleImputer(strategy="mean").fit_transform(X).ravel())

**本書に載っている出力**
```
['blue' 'green' 'red']
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]
[2. 0. 2. 1.]
[1. 2. 3.]
```

## リスト3.5　LabelEncoder・OrdinalEncoder・OneHotEncoder

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

y = ["cat", "dog", "bird", "dog"]
le = LabelEncoder()                                    # 目的変数（正解ラベル）用：1 次元
print(le.fit_transform(y).tolist(), le.classes_.tolist())
oe = OrdinalEncoder(categories=[["S", "M", "L"]])      # 順序のある特徴量：順序を指定する
print(oe.fit_transform([["M"], ["S"], ["L"]]).ravel().tolist())
print(OneHotEncoder().fit_transform([["red"], ["blue"]]).toarray().tolist())

**本書に載っている出力**
```
[1, 2, 0, 2] ['bird', 'cat', 'dog']
[1.0, 0.0, 2.0]
[[0.0, 1.0], [1.0, 0.0]]
```

## リスト3.6　Pipeline と交差検証

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=5, n_informative=3,
                           random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression())
scores = cross_val_score(model, X, y, cv=5)
print(scores.round(3))
print(round(scores.mean(), 3))

**本書に載っている出力**
```
[0.75  0.6   0.575 0.6   0.7  ]
0.645
```

## リスト3.7　LinearRegression

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([[1.0], [2.0], [3.0]])
y = np.array([2.0, 3.0, 5.0])
m = LinearRegression().fit(X, y)
print(round(m.coef_[0], 4), round(m.intercept_, 4))
print(m.predict([[4.0]]).round(4))
print(round(m.score(X, y), 4))        # 決定係数 R^2

**本書に載っている出力**
```
1.5 0.3333
[6.3333]
0.9643
```

## リスト3.8　OLS・Ridge・Lasso の重み

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso

rng = np.random.default_rng(0)
X = rng.normal(size=(30, 5))
y = 3 * X[:, 0] + rng.normal(scale=0.5, size=30)    # 効くのは1列目だけ
for name, m in [("OLS", LinearRegression()), ("Ridge", Ridge(alpha=10.0)),
                ("Lasso", Lasso(alpha=0.3))]:
    m.fit(X, y)
    print(name, m.coef_.round(3) + 0.0)

**本書に載っている出力**
```
OLS [ 2.986  0.096 -0.108 -0.248  0.029]
Ridge [ 1.594  0.231 -0.1   -0.483 -0.067]
Lasso [ 2.443  0.     0.    -0.085  0.   ]
```

## リスト3.9　LogisticRegression

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y = np.array([0, 0, 1, 0, 1, 1])
m = LogisticRegression().fit(X, y)
print(m.predict([[1.0], [6.0]]))
print(m.predict_proba([[1.0], [3.5], [6.0]])[:, 1].round(4))
z = m.decision_function([[6.0]])[0]               # w*x + b
print(round(1 / (1 + np.exp(-z)), 4))             # シグモイドを通すと確率

**本書に載っている出力**
```
[0 1]
[0.1281 0.5    0.8719]
0.8719
```

## リスト3.10　KNeighborsClassifier

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

X = np.array([[0], [1], [2], [3], [4], [5]])
y = np.array([0, 0, 1, 0, 1, 1])
for k in [1, 3]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    print(k, m.predict([[2.6]]))

**本書に載っている出力**
```
1 [0]
3 [1]
```

## リスト3.11　DecisionTreeClassifier（深さ 1）

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

X = np.array([[1], [2], [3], [4], [5], [6]])
y = np.array([0, 0, 0, 1, 0, 1])
t = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print(export_text(t, feature_names=["x"]))
print(t.tree_.impurity.round(4))      # ノードごとのジニ不純度

**本書に載っている出力**
```
|--- x <= 3.50
|   |--- class: 0
|--- x >  3.50
|   |--- class: 1

[0.4444 0.     0.4444]
```

## リスト3.12　決定木とランダムフォレスト

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = make_moons(n_samples=300, noise=0.35, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
t = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)          # 深さ無制限
f = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xtr, ytr)
print("tree  train/test", round(t.score(Xtr, ytr), 3), round(t.score(Xte, yte), 3))
print("forest train/test", round(f.score(Xtr, ytr), 3), round(f.score(Xte, yte), 3))

**本書に載っている出力**
```
tree  train/test 1.0 0.844
forest train/test 1.0 0.889
```

## リスト3.13　線形カーネルと RBF カーネル

In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
for kernel in ["linear", "rbf"]:
    m = SVC(kernel=kernel).fit(Xtr, ytr)
    print(kernel, round(m.score(Xte, yte), 3))

**本書に載っている出力**
```
linear 0.517
rbf 1.0
```

## リスト3.14　KMeans

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

X = np.array([[1.0], [2.0], [4.0], [11.0], [12.0], [14.0]])
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
print(np.sort(km.cluster_centers_.ravel()))
print(km.labels_[0] == km.labels_[2], km.labels_[0] == km.labels_[3])
print(round(km.inertia_, 4))

**本書に載っている出力**
```
[ 2.33333333 12.33333333]
True False
9.3333
```

## リスト3.15　PCA

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

X = np.array([[2.0, 0.0], [0.0, 2.0], [3.0, 1.0], [1.0, 3.0]])
pca = PCA(n_components=2).fit(X)
print(pca.explained_variance_.round(4))
print(pca.explained_variance_ratio_.round(4))
print(np.abs(pca.components_[0]).round(4))
Z = PCA(n_components=1).fit_transform(X)
print(Z.shape)

**本書に載っている出力**
```
[2.6667 0.6667]
[0.8 0.2]
[0.7071 0.7071]
(4, 1)
```

## リスト3.16　混同行列と各指標

In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 1, 1, 0, 0, 0, 0])
print(confusion_matrix(y_true, y_pred))     # [[TN FP] [FN TP]]
print(accuracy_score(y_true, y_pred))
print(round(precision_score(y_true, y_pred), 4))
print(recall_score(y_true, y_pred))
print(round(f1_score(y_true, y_pred), 4))

**本書に載っている出力**
```
[[4 2]
 [1 3]]
0.7
0.6
0.75
0.6667
```

## リスト3.17　正解率のワナ

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score

y_true = np.array([0] * 95 + [1] * 5)
y_pred = np.zeros(100, dtype=int)            # 常に「0」と答える
print(accuracy_score(y_true, y_pred), recall_score(y_true, y_pred))

**本書に載っている出力**
```
0.95 0.0
```

## リスト3.18　ROC-AUC

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

y = np.array([0, 0, 1, 1])
s = np.array([0.1, 0.4, 0.35, 0.8])
print(roc_auc_score(y, s))
fpr, tpr, th = roc_curve(y, s)
print(fpr)
print(tpr)

**本書に載っている出力**
```
0.75
[0.  0.  0.5 0.5 1. ]
[0.  0.5 0.5 1.  1. ]
```

## リスト3.19　MSE・RMSE・MAE・R²

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y = np.array([3.0, 5.0, 7.0, 9.0])
p = np.array([2.0, 5.0, 8.0, 9.0])
print(mean_squared_error(y, p), round(mean_squared_error(y, p) ** 0.5, 4))
print(mean_absolute_error(y, p))
print(r2_score(y, p))

**本書に載っている出力**
```
0.5 0.7071
0.5
0.9
```

## リスト3.20　GridSearchCV

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

X, y = make_classification(n_samples=150, n_features=4, n_informative=3,
                           n_redundant=0, random_state=1)
gs = GridSearchCV(SVC(), {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1]}, cv=5)
gs.fit(X, y)
print(gs.best_params_)
print(round(gs.best_score_, 3))
print(len(gs.cv_results_["params"]))        # 試した組み合わせの数

**本書に載っている出力**
```
{'C': 1, 'gamma': 0.1}
0.953
9
```